# Replication: Low-Volatility Anomaly
### optlab-research Â· Week 4 Validation

**Hypothesis:** Stocks with *low* idiosyncratic volatility outperform stocks with *high*
idiosyncratic volatility â€” the opposite of what CAPM predicts.

| | |
|---|---|
| **Primary signal** | `idio_vol_252d` â€” annualized FF3-residual std dev, 252 trading days |
| **Fallback signal** | `beta_60m` â€” 60-month market beta (Frazzini-Pedersen 2014 BAB) |
| **Sort direction** | **INVERTED**: Q1 (low vol/beta) = long, Q5 (high vol/beta) = short |
| **Universe** | `russell1000` (large/mid-cap) |
| **Portfolio** | Equal-weight quintile long-short â€” Q1 long, Q5 short |

**Literature:** Ang, Hodrick, Xing & Zhang (2006) *J. Finance* 61(1) [AHXZ];
Frazzini & Pedersen (2014) *JFE* 111(2) [BAB].


In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

import io, zipfile, urllib.request, datetime as _dt
import time
import duckdb
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from IPython.display import display

from optlab_research.backtest.engine import Backtest, BacktestConfig
from optlab_research.signals.compute import compute_signal

# â”€â”€ Connection (_CON_KEEPER lives at module level so GC never closes it) â”€â”€â”€â”€â”€â”€
try:
    from optlab_research import open_connection as _get_con
except (ImportError, AttributeError):
    from optlab.db import connect as _get_con

_CON_KEEPER = _get_con()

if isinstance(_CON_KEEPER, duckdb.DuckDBPyConnection):
    con = _CON_KEEPER
elif hasattr(_CON_KEEPER, '__enter__'):
    con = _CON_KEEPER.__enter__()
else:
    for _attr in ('_con', 'con', 'connection', '_connection', 'db', '_db'):
        _cand = getattr(_CON_KEEPER, _attr, None)
        if isinstance(_cand, duckdb.DuckDBPyConnection):
            con = _cand
            break
    else:
        raise RuntimeError(
            f"Cannot get DuckDB connection from {type(_CON_KEEPER)}.\n"
            f"Attrs: {[a for a in dir(_CON_KEEPER) if not a.startswith('__')]}"
        )

print(f"DuckDB connection ready: {type(con).__name__}")


## 0a. Patch: `idio_vol` permno type normalisation

`idio_vol_252d` returns `permno` as `List[Int64]` in some Polars versions
(a group-key type mismatch between the library function and `compute.py`).
This cell patches the library function before it is called so `compute_signal`
receives a clean `Int64` column. **Run this cell every session** â€” the patch
is in-memory only.


In [ ]:
import importlib

# Import the library module (cached in sys.modules after first import)
_iv_mod = importlib.import_module("optlab_research.signals.library.idio_vol")
_orig_idiovol = _iv_mod.compute

def _patched_idiovol(con, date, spec, universe):
    raw = _orig_idiovol(con=con, date=date, spec=spec, universe=universe)
    # Normalise permno: flatten List[Int64] -> Int64 if the library returned a list column.
    # This happens when Polars group_by returns the key as a list in some versions.
    ptype = raw.schema.get("permno")
    if ptype is not None and hasattr(ptype, "inner"):
        raw = raw.with_columns(pl.col("permno").list.first())
    return raw

_iv_mod.compute = _patched_idiovol
print("idio_vol.compute patched â€” permno List[Int64] will be flattened to Int64.")
print(f"Original fn: {_orig_idiovol}")


## 0b. Bootstrap `ff_factors_daily`

Downloads Ken French's daily Mkt-RF / SMB / HML / RF factors (~2 MB),
saves to the optlab Parquet lake, and registers a DuckDB view.
**Subsequent runs skip the download and just re-register the view.**


In [ ]:
OPTLAB_DATA   = Path(r"C:\Users\Owner\OneDrive\Desktop\School Work\Grad School Work\Options Club\optlab\data")
FF_DAILY_DIR  = OPTLAB_DATA / "ff_factors_daily"
FF_DAILY_PATH = FF_DAILY_DIR / "ff_factors_daily.parquet"

if not FF_DAILY_PATH.exists():
    print("Downloading Ken French daily factors (~2 MB)...")
    url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"
    with urllib.request.urlopen(url, timeout=30) as resp:
        raw = resp.read()
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        csv_name = next(n for n in zf.namelist() if n.upper().endswith(".CSV"))
        csv_text = zf.read(csv_name).decode("latin-1")
    lines = csv_text.splitlines()
    hdr_idx = next(i for i, ln in enumerate(lines) if "Mkt-RF" in ln)
    df = pd.read_csv(io.StringIO("\n".join(lines[hdr_idx:])), engine="python", on_bad_lines="skip")
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={df.columns[0]: "date_raw",
                             "Mkt-RF": "mktrf", "SMB": "smb", "HML": "hml", "RF": "rf"})
    df = df[df["date_raw"].astype(str).str.fullmatch(r"\d{8}")]
    df["date"] = pd.to_datetime(df["date_raw"].astype(str), format="%Y%m%d").dt.date
    for col in ["mktrf", "smb", "hml", "rf"]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100.0
    df = df[["date", "mktrf", "smb", "hml", "rf"]].dropna()
    FF_DAILY_DIR.mkdir(parents=True, exist_ok=True)
    pl.from_pandas(df).write_parquet(FF_DAILY_PATH)
    print(f"Saved {len(df):,} rows  ({df['date'].min()} -> {df['date'].max()})")
else:
    print(f"Using cached: {FF_DAILY_PATH}")

# Re-register view every session (views are not persisted across connections)
con.execute(f"""
    CREATE OR REPLACE VIEW ff_factors_daily AS
    SELECT * FROM read_parquet('{FF_DAILY_PATH.as_posix()}')
""")

info = con.execute(
    "SELECT MIN(date) AS min_date, MAX(date) AS max_date, COUNT(*) AS n_rows "
    "FROM ff_factors_daily"
).df()
display(info)
FF_DAILY_LAST = str(info["max_date"].iloc[0])[:10]
print(f"ff_factors_daily registered.  Last date: {FF_DAILY_LAST}")


## 0c. Data coverage & safe date

`IV_SAFE_END` = `min(DSF_LAST, FF_DAILY_LAST) âˆ’ 30 days`, snapped to month-end.
This ensures the full 252-day OLS window always has factor data, avoiding the
division-by-zero that occurs when the factor series ends before `crsp_dsf`.


In [ ]:
# crsp_dsf
try:
    dsf_info = con.execute(
        "SELECT MIN(CAST(date AS DATE)) AS min_date, "
        "       MAX(CAST(date AS DATE)) AS max_date, "
        "       COUNT(DISTINCT CAST(date AS DATE)) AS n_days "
        "FROM crsp_dsf"
    ).df()
    display(dsf_info)
    DSF_LAST   = str(dsf_info["max_date"].iloc[0])[:10]
    IDIOVOL_OK = True
    print(f"crsp_dsf:         ends {DSF_LAST}")
except Exception as e:
    IDIOVOL_OK = False
    DSF_LAST   = None
    print(f"crsp_dsf not available: {e}  -> BAB fallback only")

# Safe idiovol end date
if IDIOVOL_OK:
    _safe = min(_dt.date.fromisoformat(DSF_LAST),
                _dt.date.fromisoformat(FF_DAILY_LAST))
    _safe = _safe - _dt.timedelta(days=30)
    _safe = _safe.replace(day=1) - _dt.timedelta(days=1)  # snap to month-end
    IV_SAFE_END = _safe.isoformat()
    print(f"ff_factors_daily: ends {FF_DAILY_LAST}")
    print(f"IV_SAFE_END (FF-capped, -30d, month-end): {IV_SAFE_END}")
else:
    IV_SAFE_END = None

# crsp_msf
msf_info = con.execute(
    "SELECT MAX(CAST(date AS DATE)) AS max_date FROM crsp_msf"
).df()
CRSP_LAST = str(msf_info["max_date"].iloc[0])[:10]
print(f"crsp_msf (monthly): ends {CRSP_LAST}")


## 1. Signal validation: `idio_vol_252d`

Single-date cross-section check using the FF-capped safe date.

**Expected:** 500â€“1 000 valid rows (Russell 1000), right-skewed distribution,
median ~20â€“35% annualized, null rate 5â€“20%.


In [ ]:
if IDIOVOL_OK:
    CHECK_DATE = IV_SAFE_END
    print(f"Computing idio_vol_252d as of {CHECK_DATE} ...")
    try:
        sig_iv = compute_signal("idio_vol_252d", CHECK_DATE, con, n_quantiles=5)
        null_rate = sig_iv["signal_value"].null_count() / sig_iv.height
        print(f"  Rows  : {sig_iv.height:,}")
        print(f"  Nulls : {sig_iv['signal_value'].null_count():,}  ({null_rate:.1%})")
        print()
        display(sig_iv.group_by("signal_quantile").len().sort("signal_quantile").to_pandas())
        display(sig_iv["signal_value"].drop_nulls().describe().to_pandas())
    except Exception as e:
        IDIOVOL_OK = False
        print(f"idio_vol_252d failed: {e}")
        print("Setting IDIOVOL_OK = False. Proceed to BAB variant below.")
else:
    print("Skipping (crsp_dsf or ff_factors_daily unavailable).")


In [ ]:
if IDIOVOL_OK and 'sig_iv' in dir():
    vals = sig_iv["signal_value"].drop_nulls().to_numpy()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.hist(vals * 100, bins=80, color="teal", alpha=0.75, edgecolor="none")
    ax.axvline(float(np.median(vals)) * 100, color="firebrick", lw=1.8, ls="--",
               label=f"Median: {np.median(vals):.1%} ann.")
    ax.set_xlabel("Idiosyncratic volatility (annualized, %)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_title(f"idio_vol_252d  --  {CHECK_DATE}", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No idiovol signal to plot.")


## 2. Backtest: `idio_vol_252d` â€” INVERTED sort

| Signal | Long | Short |
|---|---|---|
| `momentum_12_2` | Q5 (winners) | Q1 (losers) |
| **`idio_vol_252d`** | **Q1 (low vol)** | **Q5 (high vol)** |

`long_quantile=1, short_quantile=5` encodes the inversion.
Window: 2019â€“2023 (validation). Change start to `"1990-01-01"` for full run.


In [ ]:
result_iv = None

if IDIOVOL_OK:
    IV_START = "2019-01-01"
    IV_END   = min("2023-12-31", IV_SAFE_END)

    cfg_iv = BacktestConfig(
        signal         = "idio_vol_252d",
        start          = IV_START,
        end            = IV_END,
        universe       = "russell1000",
        portfolio      = "quintile_long_short",
        weighting      = "equal",
        n_quantiles    = 5,
        long_quantile  = 1,   # INVERTED: Q1 (low vol) = long
        short_quantile = 5,   # INVERTED: Q5 (high vol) = short
    )
    print(f"Signal  : {cfg_iv.signal}  |  INVERTED long=Q1, short=Q5")
    print(f"Period  : {IV_START} -> {IV_END}")
    t0 = time.time()
    print("\nRunning ...")
    result_iv = Backtest(cfg_iv).run(con)
    print(f"Done in {(time.time()-t0)/60:.1f} min  ({result_iv.returns.height} months)")
else:
    print("Skipping idiovol backtest â€” see BAB below.")


In [ ]:
def _show_summary(result):
    s = result.summary()
    rows = [
        ("Period",                 f"{s['start']} -> {s['end']}"),
        ("Months",                 s["n_months"]),
        ("Avg N (long / short)",   f"{s['avg_n_long']:.0f} / {s['avg_n_short']:.0f}"),
        ("Ann. Return  L/S",       f"{s['ann_return_ls']:.2%}"        if s["ann_return_ls"]        else "N/A"),
        ("Ann. Vol     L/S",       f"{s['ann_vol_ls']:.2%}"           if s["ann_vol_ls"]           else "N/A"),
        ("Sharpe       L/S",       f"{s['sharpe_ls']:.3f}"            if s["sharpe_ls"]            else "N/A"),
        ("Max Drawdown L/S",       f"{s['max_drawdown_ls']:.2%}"      if s["max_drawdown_ls"]      else "N/A"),
        ("Monthly Win Rate",        f"{s['win_rate_ls']:.2%}"          if s["win_rate_ls"]          else "N/A"),
        ("Ann. Return  Long leg",  f"{s['ann_return_long']:.2%}"      if s["ann_return_long"]      else "N/A"),
        ("Ann. Return  Short leg", f"{s['ann_return_short']:.2%}"     if s["ann_return_short"]     else "N/A"),
        ("Avg Monthly Turnover",   f"{s['avg_monthly_turnover']:.2%}" if s["avg_monthly_turnover"] else "N/A"),
    ]
    display(pd.DataFrame(rows, columns=["Metric","Value"]).set_index("Metric"))

if result_iv is not None:
    print("idio_vol_252d summary:")
    _show_summary(result_iv)
    result_iv.plot_cumulative(figsize=(13, 5)); plt.show()
    result_iv.plot_drawdown(figsize=(13, 3));  plt.show()
else:
    print("No idiovol result â€” see BAB below.")


## 3. BAB variant: `beta_60m` (Frazzini-Pedersen 2014)

Uses only **monthly** CRSP data â€” no `crsp_dsf` or `ff_factors_daily` dependency.
Runs regardless of daily coverage issues. Requires only `ff_factors_monthly`
which is already registered in your DuckDB.

**Sort direction: INVERTED** â€” Q1 (low beta) = long, Q5 (high beta) = short.


In [ ]:
result_bab = None

cfg_bab = BacktestConfig(
    signal         = "beta_60m",
    start          = "2019-01-01",
    end            = "2023-12-31",
    universe       = "russell1000",
    portfolio      = "quintile_long_short",
    weighting      = "equal",
    n_quantiles    = 5,
    long_quantile  = 1,   # INVERTED: Q1 (low beta) = long
    short_quantile = 5,   # INVERTED: Q5 (high beta) = short
)
print(f"Signal  : {cfg_bab.signal}  |  INVERTED long=Q1, short=Q5")
print(f"Period  : {cfg_bab.start} -> {cfg_bab.end}")

try:
    t0 = time.time()
    print("\nRunning BAB backtest ...")
    result_bab = Backtest(cfg_bab).run(con)
    print(f"Done in {(time.time()-t0)/60:.1f} min  ({result_bab.returns.height} months)")
except Exception as e:
    print(f"BAB failed: {e}")


In [ ]:
if result_bab is not None:
    print("beta_60m (BAB) summary:")
    _show_summary(result_bab)
    result_bab.plot_cumulative(figsize=(13, 5)); plt.show()
    result_bab.plot_drawdown(figsize=(13, 3));  plt.show()


## 4. Side-by-side comparison

In [ ]:
rows = []
for name, res in [("idio_vol_252d (AHXZ)", result_iv), ("beta_60m (BAB)", result_bab)]:
    if res is None:
        rows.append({"Signal":name,"Period":"N/A","Ann Ret":"N/A",
                     "Sharpe":"N/A","Max DD":"N/A","Win Rate":"N/A","Turnover":"N/A"})
        continue
    s = res.summary()
    rows.append({
        "Signal"  : name,
        "Period"  : f"{s['start'][:7]} -> {s['end'][:7]}",
        "Ann Ret" : f"{s['ann_return_ls']:.2%}"        if s["ann_return_ls"]        else "N/A",
        "Sharpe"  : f"{s['sharpe_ls']:.3f}"            if s["sharpe_ls"]            else "N/A",
        "Max DD"  : f"{s['max_drawdown_ls']:.2%}"      if s["max_drawdown_ls"]      else "N/A",
        "Win Rate": f"{s['win_rate_ls']:.2%}"          if s["win_rate_ls"]          else "N/A",
        "Turnover": f"{s['avg_monthly_turnover']:.2%}" if s["avg_monthly_turnover"] else "N/A",
    })
display(pd.DataFrame(rows).set_index("Signal"))


## 5. Notes & known discrepancies

| Difference | Impact |
|---|---|
| AHXZ (2006): 1963â€“2000 full CRSP universe (micro-caps included) | Anomaly strongest in small-caps â€” large-cap only (Russell 1000) will be weaker |
| Post-2000 period: anomaly documented to have weakened (Hou & Loh 2016) | Lower Sharpe expected vs AHXZ headline |
| Short window 2019â€“2023: only 60 months | High variance â€” do not over-interpret; use for engine validation only |
| `idio_vol_252d` permno patch: `list.first()` normalisation | Cosmetic fix; signal values unaffected |

**Permanent fix (optional):** open `optlab_research/signals/library/idio_vol.py`,
find the line that builds the return DataFrame, and ensure `permno` is scalar
(e.g. call `.list.first()` or `.cast(pl.Int64)` directly in the library).

**Week 5:** Add `tcost_bps=15` â€” high-vol stocks have wide spreads, idiovol L/S
degrades meaningfully net of costs.  
**Week 6:** Expect near-zero `mktrf` loading (market-neutral L/S) in attribution.


In [ ]:
for name, res, path in [
    ("idio_vol_252d", result_iv,  "outputs/02_low_vol_idiovol"),
    ("beta_60m",      result_bab, "outputs/02_low_vol_bab"),
]:
    if res is not None:
        res.save(path)
        print(f"Saved: {path}/")
    else:
        print(f"Skipped (no result): {name}")
